## 📂 Importing necessary libraries


Fork from https://www.kaggle.com/code/zulqarnainali/fork-of-lb-0-57/notebook

In [ ]:
# 📂 Operating System and Garbage Collection
import os
import gc

# 📷 Computer Vision Library for image processing
import cv2

# 📊 Mathematical functions
import math

# 📦 Python's built-in library for shallow and deep copy operations
import copy

# ⏰ Time-related functions
import time

# 🎲 Random number generation
import random

# 🔍 File globbing utility
import glob

# 🖼️ Image processing library
from PIL import Image

# 📊 Data manipulation library
import numpy as np
import pandas as pd

# 🚀 Deep Learning Library - PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torch.cuda import amp
import torchvision

# 🛠️ Utility functions
import joblib
from tqdm import tqdm
from collections import defaultdict

# 🧱 Sklearn - Machine learning library
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

# 🌐 Image Models from torchvision and timm
import timm

# 🌈 Augmentation library for image data
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 🎨 Colored terminal text
from colorama import Fore, Back, Style
b_ = Fore.BLUE
sr_ = Style.RESET_ALL

# ⚠️ Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# 🚨 For descriptive error messages during CUDA operations
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"


In [ ]:
# 🛠️ Configuration parameters for the model and training process.

CONFIG = {
    "seed": 40,  # 🌱 Seed for reproducibility
    "img_size": 2054,  # 🖼️ Image size for training
    "model_name": "tf_efficientnetv2_s_in21ft1k",  # 🧠 Model architecture name
    "num_classes": 5,  # 🎯 Number of output classes
    "valid_batch_size": 4,  # 🚚 Batch size for validation
    "device": torch.device("cuda:0" if torch.cuda.is_available() else "cpu"),  # 🧭 Device for training (GPU if available, else CPU)
}


In [ ]:
# 🌱 Function to set seed for random number generators, ensuring reproducibility.

def set_seed(seed=42):
    np.random.seed(seed)  # 🎲 Set seed for NumPy
    torch.manual_seed(seed)  # 🚀 Set seed for PyTorch on CPU
    torch.cuda.manual_seed(seed)  # 🚀 Set seed for PyTorch on GPU
    
    # ⚙️ When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # 🔏 Set a fixed value for the hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)
    
# 🌱 Set seed using the configured seed value
set_seed(CONFIG['seed'])


In [ ]:
# 🗂️ Directory paths and file locations for data and model-related files.

ROOT_DIR = '/kaggle/input/UBC-OCEAN'  # 📁 Root directory containing the dataset
TEST_DIR = '/kaggle/input/UBC-OCEAN/test_thumbnails'  # 📁 Test thumbnails directory
ALT_TEST_DIR = '/kaggle/input/UBC-OCEAN/test_images'  # 📁 Alternative test images directory
Model_predict = '/kaggle/input/check-ponit007'  # 📁 Model prediction directory
LABEL_ENCODER_BIN = "/kaggle/input/ubcpytorchwith-classweights-training-fold1of5/label_encoder.pkl"  # 📄 Label encoder binary file
BEST_WEIGHT = "/kaggle/input/baseline-0-36/Acc0.70_Loss1.0140_epoch29_tf_efficientnetv2_s_in21ft1k_0.36.bin"  # 📄 Best weight file 1
BEST_WEIGHT2 = "/kaggle/input/ubc-efficienetnetb0-fold1of10-2048pix-thumbnails/Recall0.9178_Acc0.9437_Loss0.1685_epoch9.bin"  # 📄 Best weight file 2
BEST_WEIGHT3 = "/kaggle/input/ubc-efficienetnetb0-fold1of10-2048pix-thumbnails/Recall0.8858_Acc0.9155_Loss0.2106_epoch1.bin"  # 📄 Best weight file 3
BEST_WEIGHT4 = "/kaggle/input/ver-21-10/Acc0.50_Loss1.2095_epoch4.bin"  # 📄 Best weight file 4


In [ ]:
# 📄 Function to get the file path for a test image given its ID.

def get_test_file_path(image_id):
    # 📄 Check if the thumbnail file exists in the primary test directory
    if os.path.exists(f"{TEST_DIR}/{image_id}_thumbnail.png"):
        return f"{TEST_DIR}/{image_id}_thumbnail.png"  # 📄 Return the thumbnail file path
    else:
        return f"{ALT_TEST_DIR}/{image_id}.png"  # 📄 Return the alternative image file path


In [ ]:
# 📊 Reading test data from a CSV file and adding additional columns.

# 📊 Read the test data CSV file into a DataFrame
df = pd.read_csv(f"{ROOT_DIR}/test.csv")

# 🖼️ Add a new column 'file_path' by applying the get_test_file_path function to the 'image_id' column
df['file_path'] = df['image_id'].apply(get_test_file_path)

# 🏷️ Add a dummy 'label' column with all values set to 0
df['label'] = 0  # 🤖 Dummy label for test data


In [ ]:
# 📊 Reading the sample submission data from a CSV file.

# 📊 Read the sample submission CSV file into a DataFrame
df_sub = pd.read_csv(f"{ROOT_DIR}/sample_submission.csv")


In [ ]:
# 📄 Loading the label encoder using joblib.

# 📄 Load the label encoder from the specified binary file
encoder = joblib.load(LABEL_ENCODER_BIN)


In [ ]:
# 📄 Function to get cropped images based on specified conditions.

def get_cropped_images(file_path, image_id, th_area=1000):
    # 🖼️ Open the image using the PIL library
    image = Image.open(file_path)
    
    # 🔄 Calculate the aspect ratio
    as_ratio = image.size[0] / image.size[1]
    
    sxs, exs, sys, eys = [], [], [], []
    
    if as_ratio >= 1.5:
        # 💡 If aspect ratio is greater than or equal to 1.5, perform cropping
        
        # 🎭 Create a mask using maximum value condition
        mask = np.max(np.array(image) > 0, axis=-1).astype(np.uint8)
        
        # 🖼️ Find connected components in the mask
        retval, labels = cv2.connectedComponents(mask)
        
        if retval >= as_ratio:
            # 🔄 Loop through connected components
            x, y = np.meshgrid(np.arange(image.size[0]), np.arange(image.size[1]))
            for label in range(1, retval):
                # 🚫 Skip small components
                area = np.sum(labels == label)
                if area < th_area:
                    continue
                
                # 🔄 Get coordinates of connected components
                xs, ys = x[labels == label], y[labels == label]
                
                # 🎯 Calculate cropping boundaries
                sx, ex = np.min(xs), np.max(xs)
                cx = (sx + ex) // 2
                crop_size = image.size[1]
                sx = max(0, cx - crop_size // 2)
                ex = min(sx + crop_size - 1, image.size[0] - 1)
                sx = ex - crop_size + 1
                sy, ey = 0, image.size[1] - 1
                
                # 📊 Append cropping boundaries to lists
                sxs.append(sx)
                exs.append(ex)
                sys.append(sy)
                eys.append(ey)
        else:
            # 🎯 If no connected components found, divide the image into equal parts
            crop_size = image.size[1]
            for i in range(int(as_ratio)):
                sxs.append(i * crop_size)
                exs.append((i + 1) * crop_size - 1)
                sys.append(0)
                eys.append(crop_size - 1)
    else:
        # 🎯 If aspect ratio is less than 1.5, use the entire image without cropping
        sxs, exs, sys, eys = [0,], [image.size[0] - 1], [0,], [image.size[1] - 1]

    # 📊 Create a DataFrame with image_id, file_path, and cropping boundaries
    df_crop = pd.DataFrame()
    df_crop["image_id"] = [image_id] * len(sxs)
    df_crop["file_path"] = [file_path] * len(sxs)
    df_crop["sx"] = sxs
    df_crop["ex"] = exs
    df_crop["sy"] = sys
    df_crop["ey"] = eys
    
    return df_crop


In [ ]:
# 🔄 Loop through each row in the 'df' DataFrame and apply the get_cropped_images function.

# 📊 Initialize an empty list to store the DataFrames returned by get_cropped_images
dfs = []

# 🔄 Loop through each row in the 'df' DataFrame
for (file_path, image_id) in zip(df["file_path"], df["image_id"]):
    # 📊 Append the DataFrame returned by get_cropped_images to the list
    dfs.append(get_cropped_images(file_path, image_id))

# 📊 Concatenate the list of DataFrames into a single DataFrame
df_crop = pd.concat(dfs)

# 🏷️ Add a dummy 'label' column with all values set to 0
df_crop["label"] = 0  # 🤖 Dummy label for cropped images


In [ ]:
# 🔄 Remove duplicate rows based on specified columns and reset the index.

# 📊 Drop duplicate rows in the 'df_crop' DataFrame based on the specified subset of columns
df_crop = df_crop.drop_duplicates(subset=["image_id", "sx", "ex", "sy", "ey"]).reset_index(drop=True)


In [ ]:
# 📦 Custom dataset class for the UBC dataset.

class UBCDataset(Dataset):
    def __init__(self, df, transforms=None):
        # 📊 Initialize the dataset with DataFrame, file names, labels, and transformations
        self.df = df
        self.file_names = df['file_path'].values
        self.labels = df['label'].values
        self.transforms = transforms
        self.sxs = df["sx"].values
        self.exs = df["ex"].values
        self.sys = df["sy"].values
        self.eys = df["ey"].values
        
    def __len__(self):
        # 🔄 Return the length of the dataset
        return len(self.df)
    
    def __getitem__(self, index):
        # 🔍 Get an item from the dataset based on the index
        
        # 📄 Get image path, cropping boundaries, and label
        img_path = self.file_names[index]
        sx, ex, sy, ey = self.sxs[index], self.exs[index], self.sys[index], self.eys[index]
        
        # 🖼️ Read and convert the image to RGB
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # 🔄 Crop the image based on cropping boundaries
        img = img[sy:ey, sx:ex, :]
        
        # 🔍 Get the label
        label = self.labels[index]
        
        # 🔄 Apply transformations if specified
        if self.transforms:
            img = self.transforms(image=img)["image"]
            
        # 🔍 Return a dictionary containing the image and label as torch tensors
        return {
            'image': img,
            'label': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
# 🔄 Data transformations dictionary for validation images.

data_transforms = {
    "valid": A.Compose([
        # 🔄 Resize images to the specified size
        A.Resize(CONFIG['img_size'], CONFIG['img_size']),
        
        # 🚀 Normalize pixel values of the image
        A.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225], 
            max_pixel_value=255.0, 
            p=1.0
        ),
        
        # 🚀 Convert the image to a PyTorch tensor
        ToTensorV2()
    ], p=1.)
}

In [ ]:
# 🚀 Implementation of Generalized Mean Pooling (GeM) layer as a PyTorch module.

class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        # 🔄 Initialize the GeM layer with parameters p and eps
        super(GeM, self).__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        # 🔍 Forward pass through the GeM layer
        return self.gem(x, p=self.p, eps=self.eps)
        
    def gem(self, x, p=3, eps=1e-6):
        # 🚀 Generalized Mean Pooling function
        return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1./p)
        
    def __repr__(self):
        # 🔤 Representation of the GeM layer
        return self.__class__.__name__ + \
                '(' + 'p=' + '{:.4f}'.format(self.p.data.tolist()[0]) + \
                ', ' + 'eps=' + str(self.eps) + ')'


In [ ]:
# 🚀 Custom model class for UBC dataset based on the EfficientNet architecture.

class UBCModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=False, checkpoint_path=None):
        # 🔄 Initialize the UBCModel with specified parameters
        super(UBCModel, self).__init__()
        
        # 🚀 Create the specified model with or without pretraining
        self.model = timm.create_model(model_name, pretrained=pretrained)

        # 📏 Get the number of input features for the linear layer
        in_features = self.model.classifier.in_features
        
        # 🔄 Replace the classifier and global pooling layers with Identity
        self.model.classifier = nn.Identity()
        self.model.global_pool = nn.Identity()
        
        # 🚀 Add Generalized Mean Pooling (GeM) layer
        self.pooling = GeM()
        
        # 📏 Linear layer for final classification
        self.linear = nn.Linear(in_features, num_classes)
        
        # 🔍 Softmax layer for probability distribution
        self.softmax = nn.Softmax(dim=1)

    def forward(self, images):
        # 🔍 Forward pass through the UBCModel
        features = self.model(images)
        pooled_features = self.pooling(features).flatten(1)
        output = self.linear(pooled_features)
        return output

# 🚀 Instantiate UBCModel instances with different weights
model = UBCModel('tf_efficientnetv2_s_in21ft1k', CONFIG['num_classes'])
model2 = UBCModel('tf_efficientnet_b0_ns', CONFIG['num_classes'])
model3 = UBCModel('tf_efficientnet_b0_ns', CONFIG['num_classes'])
model4 = UBCModel('tf_efficientnet_b0_ns', CONFIG['num_classes'])

# 📄 Load the weights into the models
model.load_state_dict(torch.load(BEST_WEIGHT))
model2.load_state_dict(torch.load(BEST_WEIGHT2))
model3.load_state_dict(torch.load(BEST_WEIGHT3))
model4.load_state_dict(torch.load(BEST_WEIGHT3))

# 🔄 Move models to the specified device (GPU if available, else CPU)
model.to(CONFIG['device'])
model2.to(CONFIG['device'])
model3.to(CONFIG['device'])
model4.to(CONFIG['device'])


In [ ]:
# 🚀 Create a DataLoader for the UBC test dataset using the UBCDataset class.

# 🚀 Instantiate the UBCDataset with the cropped DataFrame and validation transforms
test_dataset = UBCDataset(df_crop, transforms=data_transforms["valid"])

# 🚚 Create a DataLoader for the test dataset with specified batch size and other settings
test_loader = DataLoader(
    test_dataset, 
    batch_size=CONFIG['valid_batch_size'], 
    num_workers=2,  # 🔍 Number of workers for data loading
    shuffle=False,   # 🔄 Do not shuffle the data for test set
    pin_memory=True  # 🚀 Pin the memory for faster GPU data transfer
)


In [ ]:
# 🔍 Inference loop for making predictions on the test set.

# 📊 Initialize an empty list to store the predictions
preds = []

# 🔍 Iterate through the test DataLoader
with torch.no_grad():
    bar = tqdm(enumerate(test_loader), total=len(test_loader))
    for step, data in bar:
        # 🔄 Move the input images to the specified device
        images = data['image'].to(CONFIG["device"], dtype=torch.float)
        
        # 🔍 Forward pass through the models and combine the outputs
        outputs1 = model(images)
        outputs2 = model2(images)
        outputs3 = model3(images)
        outputs4 = model4(images)
        
        # 🔄 Combine the model outputs using specified weights
        outputs = 0.66 * (0.34 * outputs4 + 0.7 * outputs2) + 0.322 * (0.4 * outputs1 + 0.6 * outputs3)
        
        # 🚀 Apply softmax to obtain probability distribution
        outputs = model.softmax(outputs)
        
        # 📊 Append the predictions to the list
        preds.append(outputs.detach().cpu().numpy())

# 📊 Stack the predictions into a single NumPy array
preds = np.vstack(preds)
print(preds.shape)


In [ ]:
# 🔄 Post-processing to get final predictions from the softmax outputs.

# 📊 Create columns for each category in the DataFrame based on predictions
for i in range(preds.shape[-1]):
    df_crop[f"cat{i}"] = preds[:, i]

# 📊 Create a dictionary to store the final label for each image
dict_label = {}

# 🔍 Iterate through the DataFrame grouped by "image_id"
for image_id, gdf in df_crop.groupby("image_id"):
    # 🔄 Assign the final label as the index of the maximum value in each category
    dict_label[image_id] = np.argmax(gdf[[f"cat{i}" for i in range(preds.shape[-1])]].values.max(axis=0))

# 📊 Update the 'preds' array with the final labels for each image
preds = np.array([dict_label[image_id] for image_id in df["image_id"].values])


In [ ]:
# 📊 Inverse transform the predicted labels using the label encoder and create the submission CSV.

# 📊 Inverse transform the predicted labels using the label encoder
pred_labels = encoder.inverse_transform(preds)

# 📊 Update the 'label' column in the submission DataFrame
df_sub["label"] = pred_labels

# 📄 Save the submission DataFrame to a CSV file
df_sub.to_csv("submission.csv", index=False)
